In [ ]:
"""
=============================================================================
LATIN SCRIPT ANALYSIS: EN-DE and EN-ES
=============================================================================

ANALYSES COMPUTED (no GPU needed — pure text/tokenizer-level):
 1. Word-length distribution (mean chars, median, %>10, %>15 chars)
 2. Word-length → token-count Spearman correlation (mechanistic evidence)
 3. Single-token vocabulary coverage (% word TYPES encoded as 1 XLM-R token)
 4. MATTR (morphological complexity proxy, links to Hirak et al. 2026)
 5. Byte premium (computed from actual byte counts vs English source)
 6. Compound-word proxy for German (% words needing 3+ tokens)
 7. Unified summary table across both language pairs

INPUTS:
 - tp_ip_ende_clean.csv
 - tp_ip_enes_clean.csv

OUTPUTS (saved to ../../results/extra_metrics/):
 - latin_word_length_distribution.csv
 - latin_word_token_correlation.csv
 - latin_single_token_coverage.csv
 - latin_mattr.csv
 - latin_byte_premium.csv
 - latin_compound_proxy.csv
 - latin_MASTER_SUMMARY.csv
=============================================================================
"""

import unicodedata
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from collections import Counter
import warnings
import os

warnings.filterwarnings("ignore")

OUT = "../../results/extra_metrics/"
os.makedirs(OUT, exist_ok=True)


# =============================================================================
# DEPENDENT-VOWEL UNICODE SETS
# Derived at runtime from the Unicode Character Database rather than
# hardcoded codepoint literals.  A codepoint is included when ALL of:
#   (a) its Unicode category is Mn (non-spacing mark) or Mc (spacing mark)
#   (b) its Unicode block matches the target script
#   (c) its name contains "VOWEL SIGN" or equals "VIRAMA" / "ANUSVARA" /
#       "VISARGA" / "CHILLU" — the sub-word units that XLM-R BPE isolates
#       as standalone tokens (cf. MorphTok, Brahma et al. 2025, Table 7).
# This means the sets automatically extend if Unicode adds new characters;
# no manual codepoint list maintenance is required.
# =============================================================================

_SCRIPT_BLOCKS: dict[str, tuple[int, int]] = {
    "Devanagari": (0x0900, 0x097F),   # Hindi, Marathi
    "Gujarati":   (0x0A80, 0x0AFF),
    "Tamil":      (0x0B80, 0x0BFF),
    "Malayalam":  (0x0D00, 0x0D7F),
}

_DEP_VOWEL_NAME_KEYWORDS = (
    "VOWEL SIGN",
    "VIRAMA",
    "ANUSVARA",
    "VISARGA",
    "CHILLU",
)


def _build_dep_vowel_set(block_start: int, block_end: int) -> frozenset[str]:
    """Return the set of dependent-vowel / diacritic characters for one
    Unicode block, derived from Unicode character properties."""
    result: set[str] = set()
    for cp in range(block_start, block_end + 1):
        ch = chr(cp)
        cat = unicodedata.category(ch)
        if cat not in ("Mn", "Mc"):
            continue
        try:
            name = unicodedata.name(ch)
        except ValueError:
            continue
        if any(kw in name for kw in _DEP_VOWEL_NAME_KEYWORDS):
            result.add(ch)
    return frozenset(result)


DEP_VOWELS: dict[str, frozenset[str]] = {
    script: _build_dep_vowel_set(start, end)
    for script, (start, end) in _SCRIPT_BLOCKS.items()
}


# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def parse_token_list(token_str: str) -> list[str]:
    """Convert a pipe-separated XLM-R token string to a list of tokens."""
    if pd.isna(token_str) or str(token_str).strip() == "":
        return []
    return [t.strip() for t in str(token_str).split("|") if t.strip()]


def word_token_pairs(token_list: list[str]) -> list[tuple[str, int]]:
    """
    Reconstruct (word_surface, n_tokens) pairs from an XLM-R token list.
    XLM-R uses U+2581 (▁) as a word-start prefix (SentencePiece convention).
    """
    pairs: list[tuple[str, int]] = []
    current: list[str] = []

    for tok in token_list:
        if tok.startswith("▁") and current:
            surface = "".join(current).replace("▁", "").strip()
            if surface:
                pairs.append((surface, len(current)))
            current = [tok]
        elif not current:
            current = [tok]
        else:
            current.append(tok)

    if current:
        surface = "".join(current).replace("▁", "").strip()
        if surface:
            pairs.append((surface, len(current)))

    return pairs


def compute_mattr(texts: pd.Series, window: int = 500) -> float:
    """
    Moving Average Type-Token Ratio (Covington & McFall, 2010).
    Tokenised by Unicode whitespace on lowercased surface forms.
    window=500 matches Kettunen (2014) and Hirak et al. (2026).
    """
    all_words: list[str] = [
        w
        for t in texts
        if pd.notna(t)
        for w in str(t).lower().split()
    ]
    n = len(all_words)
    if n < window:
        return len(set(all_words)) / n if n > 0 else 0.0
    return float(
        np.mean([
            len(set(all_words[i : i + window])) / window
            for i in range(n - window + 1)
        ])
    )


def mean_bytes_per_word(texts: pd.Series) -> float:
    """Mean UTF-8 byte count per whitespace-delimited word token."""
    bpw = [
        len(w.encode("utf-8"))
        for t in texts
        if pd.notna(t)
        for w in str(t).split()
    ]
    return float(np.mean(bpw)) if bpw else float("nan")


# =============================================================================
# LOAD DATA
# =============================================================================

DATA_DIR = "../../data/processed/"

print("=" * 65)
print("LOADING DATA")
print("=" * 65)

de = pd.read_csv(DATA_DIR + "tp_ip_ende_clean.csv")
es = pd.read_csv(DATA_DIR + "tp_ip_enes_clean.csv", encoding="utf-8", on_bad_lines="skip")

print(f"EN-DE: {de.shape[0]:,} rows")
print(f"EN-ES: {es.shape[0]:,} rows")

datasets: dict[str, pd.DataFrame] = {
    "EN-DE (German)": de,
    "EN-ES (Spanish)": es,
}


# =============================================================================
# ANALYSIS 1: WORD-LENGTH DISTRIBUTION
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 1: WORD-LENGTH DISTRIBUTION (target text)")
print("=" * 65)

wl_rows = []
_PUNCT = '.,;:!?()[]{}"\'\u201c\u201d\u201e\u00ab\u00bb'

for lang, df in datasets.items():
    char_lengths = [
        len(word.strip(_PUNCT))
        for text in df["target"].dropna()
        for word in str(text).split()
        if word.strip(_PUNCT)
    ]
    arr = np.array(char_lengths)
    row = {
        "Language": lang,
        "N_words": len(arr),
        "Mean_chars": round(float(np.mean(arr)), 3),
        "Median_chars": float(np.median(arr)),
        "Std_chars": round(float(np.std(arr)), 3),
        "P75_chars": float(np.percentile(arr, 75)),
        "Pct_gt10": round(100 * float(np.mean(arr > 10)), 2),
        "Pct_gt15": round(100 * float(np.mean(arr > 15)), 2),
        "Pct_gt20": round(100 * float(np.mean(arr > 20)), 2),
        "Max_chars": int(np.max(arr)),
    }
    wl_rows.append(row)
    print(f"\n  {lang}")
    print(f"    Mean word length (chars): {row['Mean_chars']}")
    print(f"    Median:                   {row['Median_chars']}")
    print(f"    % words > 10 chars:       {row['Pct_gt10']}%")
    print(f"    % words > 15 chars:       {row['Pct_gt15']}%")

pd.DataFrame(wl_rows).to_csv(OUT + "latin_word_length_distribution.csv", index=False)
print("\n  ✓ Saved: latin_word_length_distribution.csv")


# =============================================================================
# ANALYSIS 2: WORD-LENGTH → TOKEN-COUNT SPEARMAN CORRELATION
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 2: WORD-LENGTH → TOKEN-COUNT CORRELATION")
print("=" * 65)

corr_rows = []

for lang, df in datasets.items():
    char_lens, tok_counts = [], []
    for token_str in df["target_xlmr_tokens"].dropna():
        for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
            if 1 <= len(surface) <= 60:
                char_lens.append(len(surface))
                tok_counts.append(n_tok)

    rho, pval = spearmanr(char_lens, tok_counts)
    row = {
        "Language": lang,
        "N_word_tokens": len(char_lens),
        "Spearman_rho": round(rho, 4),
        "p_value": float(f"{pval:.2e}"),
        "Mean_tok_short_1to5": round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 1 <= c <= 5])), 3),
        "Mean_tok_medium_6to10": round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 6 <= c <= 10])), 3),
        "Mean_tok_long_11plus": round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if c >= 11])), 3),
    }
    corr_rows.append(row)
    print(f"\n  {lang}")
    print(f"    Spearman ρ (char_length vs n_tokens): {rho:.4f}  p={pval:.2e}")
    print(f"    Mean tokens — short (1-5):   {row['Mean_tok_short_1to5']}")
    print(f"    Mean tokens — medium (6-10): {row['Mean_tok_medium_6to10']}")
    print(f"    Mean tokens — long (11+):    {row['Mean_tok_long_11plus']}")

pd.DataFrame(corr_rows).to_csv(OUT + "latin_word_token_correlation.csv", index=False)
print("\n  ✓ Saved: latin_word_token_correlation.csv")


# =============================================================================
# ANALYSIS 3: SINGLE-TOKEN VOCABULARY COVERAGE
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 3: SINGLE-TOKEN VOCABULARY COVERAGE")
print("=" * 65)

stc_rows = []

for lang, df in datasets.items():
    word_tok_dict: dict[str, list[int]] = {}
    for token_str in df["target_xlmr_tokens"].dropna():
        for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
            key = surface.lower().strip()
            if len(key) >= 2:
                word_tok_dict.setdefault(key, []).append(n_tok)

    single = sum(
        1 for counts in word_tok_dict.values()
        if Counter(counts).most_common(1)[0][0] == 1
    )
    total = len(word_tok_dict)
    row = {
        "Language": lang,
        "Total_unique_types": total,
        "Single_token_types": single,
        "Pct_single_token": round(100 * single / total, 2),
        "Multi_token_types": total - single,
        "Pct_multi_token": round(100 * (total - single) / total, 2),
    }
    stc_rows.append(row)
    print(f"\n  {lang}")
    print(f"    Total unique word types: {total:,}")
    print(f"    Single-token types:      {single:,} ({row['Pct_single_token']}%)")
    print(f"    Multi-token types:       {row['Multi_token_types']:,} ({row['Pct_multi_token']}%)")

pd.DataFrame(stc_rows).to_csv(OUT + "latin_single_token_coverage.csv", index=False)
print("\n  ✓ Saved: latin_single_token_coverage.csv")


# =============================================================================
# ANALYSIS 4: MATTR (window=500)
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 4: MATTR (window=500)")
print("=" * 65)

mattr_rows = []

for lang, df in datasets.items():
    mattr_val = compute_mattr(df["target"].dropna())
    all_words = [w for t in df["target"].dropna() for w in str(t).lower().split()]
    simple_ttr = len(set(all_words)) / len(all_words) if all_words else 0.0
    row = {
        "Language": lang,
        "MATTR_w500": round(mattr_val, 4),
        "Simple_TTR": round(simple_ttr, 4),
        "N_tokens_total": len(all_words),
        "N_unique_types": len(set(all_words)),
    }
    mattr_rows.append(row)
    print(f"\n  {lang}")
    print(f"    MATTR (window=500): {mattr_val:.4f}")
    print(f"    Simple TTR:         {simple_ttr:.4f}")

pd.DataFrame(mattr_rows).to_csv(OUT + "latin_mattr.csv", index=False)
print("\n  ✓ Saved: latin_mattr.csv")


# =============================================================================
# ANALYSIS 5: BYTE PREMIUM vs English source
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 5: BYTE PREMIUM vs English source")
print("=" * 65)

bp_rows = []

for lang, df in datasets.items():
    tgt_mean = mean_bytes_per_word(df["target"].dropna())
    src_mean = mean_bytes_per_word(df["source"].dropna())
    premium = tgt_mean / src_mean if src_mean else float("nan")
    row = {
        "Language": lang,
        "Mean_bytes_per_word_target": round(tgt_mean, 4),
        "Mean_bytes_per_word_source": round(src_mean, 4),
        "Byte_premium": round(premium, 4),
    }
    bp_rows.append(row)
    print(f"\n  {lang}")
    print(f"    Mean bytes/word target: {tgt_mean:.4f}")
    print(f"    Mean bytes/word source: {src_mean:.4f}")
    print(f"    Byte premium:           {premium:.4f}")

pd.DataFrame(bp_rows).to_csv(OUT + "latin_byte_premium.csv", index=False)
print("\n  ✓ Saved: latin_byte_premium.csv")


# =============================================================================
# ANALYSIS 6: COMPOUND-WORD PROXY (German vs Spanish)
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 6: COMPOUND-WORD PROXY (German vs Spanish)")
print("=" * 65)

comp_rows = []

for lang, df in datasets.items():
    buckets: dict[str | int, int] = {1: 0, 2: 0, "3-5": 0, "6+": 0}
    total = 0
    for token_str in df["target_xlmr_tokens"].dropna():
        for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
            if len(surface) >= 3:
                total += 1
                if n_tok == 1:   buckets[1] += 1
                elif n_tok == 2: buckets[2] += 1
                elif n_tok <= 5: buckets["3-5"] += 1
                else:            buckets["6+"] += 1

    row = {
        "Language": lang,
        "Total_words": total,
        "Pct_1_token": round(100 * buckets[1] / total, 2),
        "Pct_2_tokens": round(100 * buckets[2] / total, 2),
        "Pct_3to5_tokens": round(100 * buckets["3-5"] / total, 2),
        "Pct_6plus_tokens": round(100 * buckets["6+"] / total, 2),
        "Pct_compound_proxy": round(100 * (buckets["3-5"] + buckets["6+"]) / total, 2),
    }
    comp_rows.append(row)
    print(f"\n  {lang}")
    print(f"    % words → 1 token:   {row['Pct_1_token']}%")
    print(f"    % words → 2 tokens:  {row['Pct_2_tokens']}%")
    print(f"    % words → 3-5 tokens:{row['Pct_3to5_tokens']}%  ← compound proxy")
    print(f"    % words → 6+ tokens: {row['Pct_6plus_tokens']}%  ← heavy fragmentation")
    print(f"    % compound proxy (3+):{row['Pct_compound_proxy']}%")

pd.DataFrame(comp_rows).to_csv(OUT + "latin_compound_proxy.csv", index=False)
print("\n  ✓ Saved: latin_compound_proxy.csv")


# =============================================================================
# ANALYSIS 7: TP/IP SUMMARY FROM EXISTING DATASET COLUMNS
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 7: TP/IP SUMMARY FROM EXISTING DATASET COLUMNS")
print("=" * 65)

tp_ip_rows = []

for lang, df in datasets.items():
    row = {
        "Language": lang,
        "Mean_IP": round(df["target_xlmr_IP"].mean(), 4),
        "Std_IP": round(df["target_xlmr_IP"].std(), 4),
        "Mean_TP": round(df["target_xlmr_TP"].mean(), 4),
        "Std_TP": round(df["target_xlmr_TP"].std(), 4),
        "Mean_COMET": round(df["COMET"].mean(), 4),
        "IP_COMET_corr": round(df[["target_xlmr_IP", "COMET"]].corr().iloc[0, 1], 4),
        "TP_COMET_corr": round(df[["target_xlmr_TP", "COMET"]].corr().iloc[0, 1], 4),
        "N_segments": len(df),
    }
    tp_ip_rows.append(row)
    print(f"\n  {lang}")
    for k, v in row.items():
        if k != "Language":
            print(f"    {k}: {v}")


# =============================================================================
# MASTER SUMMARY TABLE
# =============================================================================

print("\n" + "=" * 65)
print("MASTER SUMMARY TABLE")
print("=" * 65)

lang_order = ["EN-ES (Spanish)", "EN-DE (German)"]
script_map = {"EN-ES (Spanish)": "Latin/Romance", "EN-DE (German)": "Latin/Germanic"}

tp_ip_dict  = {r["Language"]: r for r in tp_ip_rows}
mattr_dict  = {r["Language"]: r for r in mattr_rows}
bp_dict     = {r["Language"]: r for r in bp_rows}
stc_dict    = {r["Language"]: r for r in stc_rows}
comp_dict   = {r["Language"]: r for r in comp_rows}
corr_dict   = {r["Language"]: r for r in corr_rows}

summary_rows = []
for lang in lang_order:
    summary_rows.append({
        "Language":            lang,
        "Script":              script_map[lang],
        "Mean_IP":             tp_ip_dict[lang]["Mean_IP"],
        "Mean_TP":             tp_ip_dict[lang]["Mean_TP"],
        "MATTR":               mattr_dict[lang]["MATTR_w500"],
        "Byte_premium":        bp_dict[lang]["Byte_premium"],
        "Pct_single_token":    stc_dict[lang]["Pct_single_token"],
        "Pct_compound_3plus":  comp_dict[lang]["Pct_compound_proxy"],
        "WL_tok_spearman_rho": corr_dict[lang]["Spearman_rho"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT + "latin_MASTER_SUMMARY.csv", index=False)
pd.DataFrame(tp_ip_rows).to_csv(OUT + "latin_tpip_stats.csv", index=False)

print("\n" + summary_df.to_string(index=False))
print("\n  ✓ Saved: latin_MASTER_SUMMARY.csv")

print("\n" + "=" * 65)
print("ALL ANALYSES COMPLETE — files saved to:", OUT)
print("=" * 65)


In [ ]:
"""
=============================================================================
INDIC LANGUAGE ANALYSIS: Hindi, Tamil, Malayalam, Gujarati, Marathi
=============================================================================

ANALYSES COMPUTED (no GPU needed — pure text/token-level analysis):
 1. Dependent vowel fragmentation (isolated diacritic tokens per language)
 2. Romanisation decomposition (char overhead, token reduction, Computational Tax)
 3. MATTR native and romanised (links to Hirak et al. 2026, Manohar et al. 2020)
 4. Word-length → token-count Spearman ρ (native + romanised)
 5. Single-token vocabulary coverage (native vs romanised)
 6. Byte premium vs English source
 7. Cross-language master comparison table

INPUTS:
 - ../../data/processed/<language>_indicmt.csv  (one per language)

OUTPUTS (saved to ../../results/extra_metrics/):
 - indic_dependent_vowel_fragmentation.csv
 - indic_romanisation_decomposition.csv
 - indic_mattr_native_roman.csv
 - indic_word_token_correlation.csv
 - indic_single_token_coverage.csv
 - indic_byte_premium.csv
 - indic_tpip_stats.csv
 - indic_MASTER_SUMMARY.csv
=============================================================================
"""

# DEP_VOWELS, all utility functions, and OUT are inherited from Cell 1.

# =============================================================================
# LANGUAGE CONFIGURATION
# =============================================================================

# Script assignment used only for the summary table; detection logic
# is driven by DEP_VOWELS which is derived from Unicode properties above.
LANGUAGE_META: dict[str, dict] = {
    "gujarati":  {"display": "Gujarati",  "script": "Gujarati",    "dep_vowel_key": "Gujarati"},
    "hindi":     {"display": "Hindi",     "script": "Devanagari",  "dep_vowel_key": "Devanagari"},
    "marathi":   {"display": "Marathi",   "script": "Devanagari",  "dep_vowel_key": "Devanagari"},
    "tamil":     {"display": "Tamil",     "script": "Tamil",       "dep_vowel_key": "Tamil"},
    "malayalam": {"display": "Malayalam", "script": "Malayalam",   "dep_vowel_key": "Malayalam"},
}

# Column names — defined once, referenced throughout.
COL_HYP      = "Translation"
COL_REF      = "Reference"
COL_SRC      = "Source"
COL_HYP_ROM  = "Translation_Transliteration_romanized"
COL_REF_ROM  = "Reference_Transliteration_romanized"
COL_IP       = "target_xlmr_IP"
COL_TP       = "target_xlmr_TP"
COL_TOKENS   = "target_xlmr_tokens"
COL_COMET    = "COMET"

# Display order in output tables (low → high fragmentation).
LANG_ORDER = ["gujarati", "hindi", "marathi", "tamil", "malayalam"]


# =============================================================================
# LOAD DATA
# =============================================================================

DATA_DIR = "../../data/processed/"

print("=" * 65)
print("LOADING INDIC DATA")
print("=" * 65)

indic: dict[str, pd.DataFrame] = {}
for lang in LANG_ORDER:
    path = f"{DATA_DIR}{lang}_indicmt.csv"
    df = pd.read_csv(path)
    indic[lang] = df
    meta = LANGUAGE_META[lang]
    print(f"  {meta['display']:12s}: {df.shape[0]:,} rows, {df.shape[1]} cols")

print(f"\n  Analysis order (low→high fragmentation): {LANG_ORDER}")


# =============================================================================
# ANALYSIS 1: DEPENDENT VOWEL FRAGMENTATION
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 1: DEPENDENT VOWEL (DIACRITIC) FRAGMENTATION")
print("=" * 65)
print("  Counts isolated dependent-vowel tokens in XLM-R output.")
print("  Diacritic set derived from Unicode properties (see Cell 1).")

dv_rows = []

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]
    dv_set = DEP_VOWELS[meta["dep_vowel_key"]]

    dv_count = 0
    total_tokens = 0
    sentences_with_dv = 0

    for token_str in df[COL_TOKENS].dropna():
        tokens = parse_token_list(token_str)
        total_tokens += len(tokens)
        # A dependent-vowel token: stripped of the SentencePiece prefix,
        # the entire remaining string consists solely of diacritic codepoints.
        isolated = [
            t for t in tokens
            if all(ch in dv_set for ch in t.lstrip("▁")) and t.lstrip("▁")
        ]
        dv_count += len(isolated)
        if isolated:
            sentences_with_dv += 1

    n_sentences = df[COL_TOKENS].dropna().shape[0]
    rate_per_1k = round(1000 * dv_count / total_tokens, 3) if total_tokens else float("nan")

    row = {
        "Language":             meta["display"],
        "Script":               meta["script"],
        "N_DV_tokens":          dv_count,
        "Total_tokens":         total_tokens,
        "DV_rate_per_1k_tokens": rate_per_1k,
        "N_sentences_with_DV": sentences_with_dv,
        "Pct_sentences_with_DV": round(100 * sentences_with_dv / n_sentences, 1),
        "Mean_IP": round(df[COL_IP].mean(), 4),
        "Mean_TP": round(df[COL_TP].mean(), 4),
    }
    dv_rows.append(row)
    print(f"\n  {meta['display']} ({meta['script']})")
    print(f"    Isolated dep vowel tokens:   {dv_count:,}")
    print(f"    Rate per 1,000 tokens:       {rate_per_1k}")
    print(f"    Sentences with any DV tok:   {sentences_with_dv:,} ({row['Pct_sentences_with_DV']}%)")
    print(f"    Mean IP: {row['Mean_IP']}    Mean TP: {row['Mean_TP']}")

pd.DataFrame(dv_rows).to_csv(OUT + "indic_dependent_vowel_fragmentation.csv", index=False)
print("\n  ✓ Saved: indic_dependent_vowel_fragmentation.csv")


# =============================================================================
# ANALYSIS 2: ROMANISATION DECOMPOSITION
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 2: ROMANISATION DECOMPOSITION")
print("  (Character overhead + Token reduction + Computational Tax)")
print("=" * 65)

rom_rows = []

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]

    native_chars  = df[COL_HYP].dropna().apply(lambda t: len(str(t))).mean()
    roman_chars   = df[COL_HYP_ROM].dropna().apply(lambda t: len(str(t))).mean()
    native_tokens = df[COL_TOKENS].dropna().apply(lambda t: len(parse_token_list(t))).mean()

    roman_tok_col = COL_TOKENS.replace("target", "target_roman")  # column written by 03
    if roman_tok_col not in df.columns:
        roman_tok_col = "roman_xlmr_tokens"
    roman_tokens = df[roman_tok_col].dropna().apply(lambda t: len(parse_token_list(t))).mean() \
        if roman_tok_col in df.columns else float("nan")

    ip_native = df[COL_IP].mean()
    comet_native = df[COL_COMET].mean()

    ip_roman_col    = COL_IP.replace("target", "roman")
    comet_roman_col = COL_COMET + "_roman"
    ip_roman    = df[ip_roman_col].mean()    if ip_roman_col    in df.columns else float("nan")
    comet_roman = df[comet_roman_col].mean() if comet_roman_col in df.columns else float("nan")

    char_overhead  = round(roman_chars   / native_chars,   4) if native_chars   else float("nan")
    token_ratio    = round(roman_tokens  / native_tokens,  4) if native_tokens  else float("nan")
    ip_change_pct  = round(100 * (ip_roman    - ip_native)    / ip_native,    2) if ip_native    else float("nan")
    comet_chg_pct  = round(100 * (comet_roman - comet_native) / comet_native, 2) if comet_native else float("nan")

    row = {
        "Language":          meta["display"],
        "Char_overhead":     char_overhead,
        "Token_ratio_roman_native": token_ratio,
        "IP_native":         round(ip_native, 4),
        "IP_roman":          round(ip_roman,  4),
        "IP_change_pct":     ip_change_pct,
        "COMET_native":      round(comet_native, 2),
        "COMET_roman":       round(comet_roman,  2),
        "COMET_change_pct":  comet_chg_pct,
    }
    rom_rows.append(row)
    sign = lambda x: f"+{x}" if x > 0 else str(x)
    print(f"\n  {meta['display']}")
    print(f"    Char overhead (roman/native chars): {char_overhead:.4f}×")
    print(f"    Token ratio   (roman/native tokens): {token_ratio:.4f}×")
    print(f"    IP  native → roman: {ip_native:.4f} → {ip_roman:.4f}  ({sign(ip_change_pct)}%)")
    print(f"    COMET native → roman: {comet_native:.2f} → {comet_roman:.2f}  ({sign(comet_chg_pct)}%)")

pd.DataFrame(rom_rows).to_csv(OUT + "indic_romanisation_decomposition.csv", index=False)
print("\n  ✓ Saved: indic_romanisation_decomposition.csv")


# =============================================================================
# ANALYSIS 3: MATTR (window=500) — native and romanised
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 3: MATTR (window=500) — native and romanised")
print("=" * 65)

mattr_indic_rows = []

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]
    mattr_native = compute_mattr(df[COL_HYP].dropna())
    mattr_roman  = compute_mattr(df[COL_HYP_ROM].dropna())
    row = {
        "Language":    meta["display"],
        "Script":      meta["script"],
        "MATTR_native": round(mattr_native, 4),
        "MATTR_roman":  round(mattr_roman,  4),
        "MATTR_delta":  round(mattr_roman - mattr_native, 4),
    }
    mattr_indic_rows.append(row)
    print(f"\n  {meta['display']} ({meta['script']})")
    print(f"    MATTR native:    {mattr_native:.4f}")
    print(f"    MATTR romanised: {mattr_roman:.4f}  (Δ = {row['MATTR_delta']:+.4f})")

pd.DataFrame(mattr_indic_rows).to_csv(OUT + "indic_mattr_native_roman.csv", index=False)
print("\n  ✓ Saved: indic_mattr_native_roman.csv")


# =============================================================================
# ANALYSIS 4: WORD-LENGTH → TOKEN-COUNT SPEARMAN ρ
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 4: WORD-LENGTH → TOKEN-COUNT SPEARMAN ρ")
print("  (native and romanised — reveals fragmentation mechanism)")
print("=" * 65)

wt_indic_rows = []

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]

    roman_tok_col = "roman_xlmr_tokens" if "roman_xlmr_tokens" in df.columns \
        else COL_TOKENS.replace("target", "target_roman")

    for suffix, tok_col in (("native", COL_TOKENS), ("roman", roman_tok_col)):
        if tok_col not in df.columns:
            continue
        char_lens, tok_counts = [], []
        for token_str in df[tok_col].dropna():
            for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
                if 1 <= len(surface) <= 60:
                    char_lens.append(len(surface))
                    tok_counts.append(n_tok)
        rho, pval = spearmanr(char_lens, tok_counts)
        row = {
            "Language":      meta["display"],
            "Script_form":   suffix,
            "N_word_tokens": len(char_lens),
            "Spearman_rho":  round(rho, 4),
            "p_value":       float(f"{pval:.2e}"),
            "Mean_tok_1to3":  round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 1 <= c <= 3])), 3),
            "Mean_tok_4to6":  round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 4 <= c <= 6])), 3),
            "Mean_tok_7to10": round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if 7 <= c <= 10])), 3),
            "Mean_tok_11plus": round(float(np.mean([n for c, n in zip(char_lens, tok_counts) if c >= 11])), 3),
        }
        wt_indic_rows.append(row)
        if suffix == "native":
            print(f"\n  {meta['display']} (native script)")
            print(f"    Spearman ρ: {rho:.4f}  p={pval:.2e}")
            print(f"    Mean tokens: 1-3={row['Mean_tok_1to3']}  4-6={row['Mean_tok_4to6']}  "
                  f"7-10={row['Mean_tok_7to10']}  11+={row['Mean_tok_11plus']}")
        else:
            print(f"  {meta['display']} (romanised)")
            print(f"    Spearman ρ: {rho:.4f}  p={pval:.2e}")

pd.DataFrame(wt_indic_rows).to_csv(OUT + "indic_word_token_correlation.csv", index=False)
print("\n  ✓ Saved: indic_word_token_correlation.csv")


# =============================================================================
# ANALYSIS 5: SINGLE-TOKEN VOCABULARY COVERAGE
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 5: SINGLE-TOKEN VOCABULARY COVERAGE")
print("  (% word types as 1 XLM-R token: native vs romanised)")
print("=" * 65)

stc_indic_rows = []

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]
    roman_tok_col = "roman_xlmr_tokens" if "roman_xlmr_tokens" in df.columns \
        else COL_TOKENS.replace("target", "target_roman")

    for suffix, tok_col in (("native", COL_TOKENS), ("roman", roman_tok_col)):
        if tok_col not in df.columns:
            continue
        word_tok_dict: dict[str, list[int]] = {}
        for token_str in df[tok_col].dropna():
            for surface, n_tok in word_token_pairs(parse_token_list(token_str)):
                key = surface.lower().strip()
                if len(key) >= 2:
                    word_tok_dict.setdefault(key, []).append(n_tok)
        single = sum(1 for c in word_tok_dict.values() if Counter(c).most_common(1)[0][0] == 1)
        total  = len(word_tok_dict)
        pct    = round(100 * single / total, 2) if total else float("nan")
        stc_indic_rows.append({
            "Language":            meta["display"],
            "Script_form":         suffix,
            "Total_unique_types":  total,
            "Single_token_types":  single,
            "Pct_single_token":    pct,
        })
        print(f"  {meta['display']} ({suffix}): {single}/{total} word types = {pct}% single-token")

pd.DataFrame(stc_indic_rows).to_csv(OUT + "indic_single_token_coverage.csv", index=False)
print("\n  ✓ Saved: indic_single_token_coverage.csv")


# =============================================================================
# ANALYSIS 6: BYTE PREMIUM vs English source
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 6: BYTE PREMIUM vs English source")
print("  Key argument: similar byte premiums → CANNOT explain within-Indic ordering")
print("=" * 65)

bp_indic_rows = []

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]
    src_bpw    = mean_bytes_per_word(df[COL_SRC].dropna())
    native_bpw = mean_bytes_per_word(df[COL_HYP].dropna())
    roman_bpw  = mean_bytes_per_word(df[COL_HYP_ROM].dropna())
    row = {
        "Language":             meta["display"],
        "Script":               meta["script"],
        "Bytes_per_word_source": round(src_bpw,    3),
        "Bytes_per_word_native": round(native_bpw, 3),
        "Bytes_per_word_roman":  round(roman_bpw,  3),
        "Byte_premium_native":   round(native_bpw / src_bpw, 3) if src_bpw else float("nan"),
        "Byte_premium_roman":    round(roman_bpw  / src_bpw, 3) if src_bpw else float("nan"),
    }
    bp_indic_rows.append(row)
    print(f"\n  {meta['display']} ({meta['script']})")
    print(f"    Bytes/word source (EN): {row['Bytes_per_word_source']}")
    print(f"    Bytes/word target native: {row['Bytes_per_word_native']}  ({row['Byte_premium_native']}×)")
    print(f"    Bytes/word target roman:  {row['Bytes_per_word_roman']}  ({row['Byte_premium_roman']}×)")

pd.DataFrame(bp_indic_rows).to_csv(OUT + "indic_byte_premium.csv", index=False)
print("\n  ✓ Saved: indic_byte_premium.csv")


# =============================================================================
# ANALYSIS 7: TP/IP STATS (verification from existing columns)
# =============================================================================

print("\n" + "=" * 65)
print("ANALYSIS 7: TP/IP STATS (verification from existing columns)")
print("=" * 65)

tp_ip_indic_rows = []

ip_roman_col    = COL_IP.replace("target", "roman")
tp_roman_col    = COL_TP.replace("target", "roman")
comet_roman_col = COL_COMET + "_roman"

for lang in LANG_ORDER:
    df = indic[lang]
    meta = LANGUAGE_META[lang]
    ip_nat   = df[COL_IP].mean()
    tp_nat   = df[COL_TP].mean()
    comet_nat = df[COL_COMET].mean()
    ip_rom   = df[ip_roman_col].mean()    if ip_roman_col    in df.columns else float("nan")
    tp_rom   = df[tp_roman_col].mean()    if tp_roman_col    in df.columns else float("nan")
    comet_rom = df[comet_roman_col].mean() if comet_roman_col in df.columns else float("nan")
    ip_comet_corr = df[[COL_IP, COL_COMET]].corr().iloc[0, 1]
    row = {
        "Language":      meta["display"],
        "IP_native":     round(ip_nat, 4),
        "IP_roman":      round(ip_rom, 4),
        "TP_native":     round(tp_nat, 4),
        "TP_roman":      round(tp_rom, 4),
        "IP_COMET_corr": round(ip_comet_corr, 4),
        "COMET_native":  round(comet_nat, 2),
        "COMET_roman":   round(comet_rom,  2),
        "COMET_delta":   round(comet_rom - comet_nat, 2),
    }
    tp_ip_indic_rows.append(row)
    print(f"\n  {meta['display']}")
    print(f"    IP  native/roman: {ip_nat:.4f} → {ip_rom:.4f}")
    print(f"    TP  native/roman: {tp_nat:.4f} → {tp_rom:.4f}")
    print(f"    IP-COMET corr:    {ip_comet_corr:.4f}")
    print(f"    COMET delta:      {comet_nat:.2f} → {comet_rom:.2f}  ({row['COMET_delta']:+.2f})")

pd.DataFrame(tp_ip_indic_rows).to_csv(OUT + "indic_tpip_stats.csv", index=False)
print("\n  ✓ Saved: indic_tpip_stats.csv")


# =============================================================================
# MASTER SUMMARY TABLE (Indic — all analyses)
# =============================================================================

print("\n" + "=" * 65)
print("MASTER SUMMARY TABLE (Indic — all analyses)")
print("=" * 65)

dv_dict    = {r["Language"]: r for r in dv_rows}
rom_dict   = {r["Language"]: r for r in rom_rows}
mattr_dict_i = {r["Language"]: r for r in mattr_indic_rows}
stc_native = {r["Language"]: r for r in stc_indic_rows if r["Script_form"] == "native"}
bp_dict_i  = {r["Language"]: r for r in bp_indic_rows}
tpip_dict  = {r["Language"]: r for r in tp_ip_indic_rows}

summary_indic_rows = []
for lang in LANG_ORDER:
    disp = LANGUAGE_META[lang]["display"]
    summary_indic_rows.append({
        "Language":              disp,
        "Script":                LANGUAGE_META[lang]["script"],
        "Mean_IP_native":        tpip_dict[disp]["IP_native"],
        "Mean_TP_native":        tpip_dict[disp]["TP_native"],
        "DV_rate_per_1k_tokens": dv_dict[disp]["DV_rate_per_1k_tokens"],
        "Char_overhead_ratio":   rom_dict[disp]["Char_overhead"],
        "MATTR_native":          mattr_dict_i[disp]["MATTR_native"],
        "Pct_single_tok_native": stc_native[disp]["Pct_single_token"],
        "Byte_premium_native":   bp_dict_i[disp]["Byte_premium_native"],
        "COMET_change_pct":      rom_dict[disp]["COMET_change_pct"],
    })

summary_indic_df = pd.DataFrame(summary_indic_rows)
summary_indic_df.to_csv(OUT + "indic_MASTER_SUMMARY.csv", index=False)
print("\n" + summary_indic_df.to_string(index=False))
print("\n  ✓ Saved: indic_MASTER_SUMMARY.csv")

print("\n" + "=" * 65)
print("ALL ANALYSES COMPLETE")
print("Files saved to:", OUT)
print("=" * 65)
